<a href="https://colab.research.google.com/github/Mutasar/Proyek_Analisis_Sentimen/blob/main/Analisa_Sentimen_Ulasan_Aplikasi_Tokopedia_di_Playstore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import dataset yang telah dilakukan text preprocessing

In [95]:

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from google.colab import drive
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.cluster import KMeans
from yellowbrick.cluster import KElbowVisualizer
import joblib
from google.colab import files


# 1. inisialisasi dataset

In [75]:
# inisialisasi dataset
file_id = '1GLyhzyN3XNgmC3ftv4Klmvtlt_Dc7XzD'
download_url = f'https://drive.google.com/uc?id={file_id}'

# Membaca CSV
df = pd.read_csv(download_url)

In [76]:
df.head()

,userName,score,at,content
0,Pengguna Google,5,2024-09-08 03:31:50,makasih toped
1,Pengguna Google,1,2024-09-08 03:29:56,Aplikasi php sudah banyak dikasih promo &sudah...
2,Pengguna Google,5,2024-09-08 03:26:38,mantab... 👍
3,Pengguna Google,5,2024-09-08 03:25:14,Good good
4,Pengguna Google,1,2024-09-08 03:24:01,Sangat buruk sebagai pengguna lama akun affali...


In [78]:
# Tinjau jumlah baris kolom dan jenis data dalam dataset dengan info.

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 5 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   userName  500000 non-null  object
 1   score     500000 non-null  int64 
 2   at        500000 non-null  object
 3   content   499999 non-null  object
 4   label     500000 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 19.1+ MB


In [79]:
# Menampilkan statistik deskriptif dataset dengan menjalankan describe

df.describe()

,score,label
count,500000.000000,500000.000000
mean,4.074004,0.759270
std,1.545313,0.427527
min,1.000000,0.000000
25%,4.000000,1.000000
50%,5.000000,1.000000
75%,5.000000,1.000000
max,5.000000,1.000000


In [91]:
df = df.rename(columns={
    'userName': 'nama',
    'score': 'rating',
    'at': 'waktu',
    'content': 'ulasan'
})
print(df.columns)

Index(['nama', 'rating', 'waktu', 'ulasan', 'label'], dtype='object')


In [92]:
df.head()

,nama,rating,waktu,ulasan,label
0,Pengguna Google,5,2024-09-08 03:31:50,makasih toped,1
1,Pengguna Google,1,2024-09-08 03:29:56,Aplikasi php sudah banyak dikasih promo &sudah...,0
2,Pengguna Google,5,2024-09-08 03:26:38,mantab... 👍,1
3,Pengguna Google,5,2024-09-08 03:25:14,Good good,1
4,Pengguna Google,1,2024-09-08 03:24:01,Sangat buruk sebagai pengguna lama akun affali...,0


# 2. Cleaning Data

In [111]:
# Mengecek dataset menggunakan isnull().sum()

print("Cek nilai null:\n", df.isnull().sum())

Cek nilai null:
 nama      0
rating    0
waktu     0
ulasan    0
label     0
dtype: int64


In [83]:
# Mengecek dataset menggunakan duplicated().sum()

print("\nJumlah duplikasi:", df.duplicated().sum())


Jumlah duplikasi: 10


In [84]:
#menghilangkan data duplikat
df = df.drop_duplicates()

In [85]:
#cek hasil drop duplikat
print("\nJumlah duplikasi:", df.duplicated().sum())


Jumlah duplikasi: 0


In [99]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [107]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [108]:
# Fungsi pembersihan teks
def clean_text(text):
    if pd.isnull(text):  # Tangani NaN
        return ""
    text = text.lower()  # Ubah ke huruf kecil
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)  # Hapus URL
    text = re.sub(r'[^a-z\s]', '', text)  # Hapus karakter non-abjad
    tokens = word_tokenize(text)  # Tokenisasi
    stop_words = set(stopwords.words('indonesian'))  # Ubah ke bahasa Indonesia jika perlu
    tokens = [word for word in tokens if word not in stop_words]  # Hapus stopwords
    return ' '.join(tokens)

# Terapkan ke kolom teks (misalnya 'ulasan')
df['ulasan'] = df['ulasan'].apply(clean_text)

In [109]:
# Tampilkan beberapa baris pertama dari kolom 'ulasan' setelah pembersihan
print("\nHasil pembersihan teks:")
print(df['ulasan'].head())


Hasil pembersihan teks:
0                                        makasih toped
1    aplikasi php dikasih promo dibayar diklik diba...
2                                               mantab
3                                            good good
4    buruk pengguna akun affalite blokir rugikan share
Name: ulasan, dtype: object


# Labeling

Pembagian data menjadi data sentimen berlabel positif dan negatif dengan angka 1 untuk positif dan angka 0 untuk negatif. Pengklasifikasian ini dilakukan pada ulasan yang memiliki rating 4 dan 5 sebagai sentimen positif dan rating 3 sampai 1 sebagai sentimen negatif.

In [119]:
# sentimen berdasarkan rating
def get_sentiment_label(rating):
    if rating >= 4:
        return 1  # Positif
    elif rating <= 3:
        return 0  # Negatif
    else:
        return None  # Jika rating tidak valid

# menerapkan fungsi ke DataFrame
df['label'] = df['rating'].apply(get_sentiment_label)

print(df.head())


              nama  rating                waktu  \
0  Pengguna Google       5  2024-09-08 03:31:50   
1  Pengguna Google       1  2024-09-08 03:29:56   
2  Pengguna Google       5  2024-09-08 03:26:38   
3  Pengguna Google       5  2024-09-08 03:25:14   
4  Pengguna Google       1  2024-09-08 03:24:01   

                                              ulasan  label  label_sentimen  
0                                      makasih toped      1               1  
1  aplikasi php dikasih promo dibayar diklik diba...      0               0  
2                                             mantab      1               1  
3                                          good good      1               1  
4  buruk pengguna akun affalite blokir rugikan share      0               0  


In [120]:
df["label"].value_counts()

,count
label,
1,379625
0,120365


# Menghitung Kata Dengan TF-IDF

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

In [ ]:
Ulasan = df['content']

In [ ]:
Ulasan.isnull().sum()

np.int64(1)

In [ ]:
Ulasan = Ulasan.fillna('tidak ada komentar')

In [ ]:
#untuk menghitung jumlah kata yang telah di steming
cv = CountVectorizer()
term_fit = cv.fit(Ulasan)

print (len(term_fit.vocabulary_))

88203


In [ ]:
term_fit.vocabulary_ #mengurutkan berdasarkan urutab abjad kata

{'makasih': 44786,
 'toped': 80599,
 'aplikasi': 6020,
 'php': 63034,
 'sudah': 74755,
 'banyak': 8827,
 'dikasih': 20342,
 'promo': 64974,
 'dibayar': 19340,
 'tapi': 76431,
 'diklik': 20517,
 'jawabnya': 34098,
 'dibatalkan': 19296,
 'sistem': 73015,
 'tolong': 80424,
 'perbaiki': 62042,
 'kmu': 39784,
 'tokopedia': 80033,
 'mantab': 45806,
 'good': 28455,
 'sangat': 68818,
 'buruk': 14821,
 'sebagai': 69623,
 'pengguna': 61383,
 'lama': 42024,
 'akun': 4119,
 'affalite': 3415,
 'di': 18946,
 'blokir': 12984,
 'tanpa': 76380,
 'sebab': 69618,
 'amat': 4694,
 'rugikan': 67866,
 'dalam': 17778,
 'hal': 29967,
 'share': 72151,
 'apa': 5676,
 'harus': 30401,
 'verifikasi': 83525,
 'nomor': 56019,
 'hp': 31544,
 'gabisa': 26565,
 'dipake': 21091,
 'beberapa': 9632,
 'udah': 82149,
 'tau': 76581,
 'ribet': 67268,
 'beli': 10055,
 'baru': 9113,
 'tuh': 81689,
 'payah': 59623,
 'diskonnya': 21976,
 'gokil': 28401,
 'oke': 57325,
 'hallo': 30007,
 'dipermudah': 21275,
 'lagi': 41827,
 'pembay

In [ ]:
#kolom pertama ini berarti jumlah dokumen
#kolom kedua berarti letak katanya
#kolom ketiga hasil dari tf

term_frequency_all = term_fit.transform(Ulasan)
print (term_frequency_all)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 3447619 stored elements and shape (500000, 88203)>
  Coords	Values
  (0, 44786)	1
  (0, 80599)	1
  (1, 6020)	1
  (1, 8827)	1
  (1, 19296)	1
  (1, 19340)	1
  (1, 20342)	1
  (1, 20517)	1
  (1, 34098)	1
  (1, 39784)	1
  (1, 62042)	1
  (1, 63034)	1
  (1, 64974)	1
  (1, 73015)	2
  (1, 74755)	2
  (1, 76431)	1
  (1, 80033)	1
  (1, 80424)	1
  (2, 45806)	1
  (3, 28455)	2
  (4, 3415)	1
  (4, 4119)	1
  (4, 4694)	1
  (4, 12984)	1
  (4, 14821)	1
  :	:
  (499995, 7734)	1
  (499995, 68818)	1
  (499996, 6020)	1
  (499996, 8827)	1
  (499996, 12571)	1
  (499996, 14439)	1
  (499996, 17824)	1
  (499996, 18608)	1
  (499996, 32821)	1
  (499996, 41918)	1
  (499996, 45344)	1
  (499996, 48179)	1
  (499996, 50068)	1
  (499996, 52825)	1
  (499996, 60391)	1
  (499996, 69213)	2
  (499996, 83079)	2
  (499996, 84821)	1
  (499996, 85837)	1
  (499997, 47960)	1
  (499997, 68818)	1
  (499998, 48580)	1
  (499998, 68818)	1
  (499999, 11156)	1
  (499999, 68818)	1


In [ ]:
ulasan_tf = Ulasan[1] #memanggil kata pada index ke 1
print (ulasan_tf)

Aplikasi php sudah banyak dikasih promo &sudah dibayar tapi diklik jawabnya dibatalkan sistem tolong perbaiki sistem kmu tokopedia


In [ ]:
term_frequency = term_fit.transform([ulasan_tf]) #hanya menampilkan hasil document 1
print (term_frequency)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 16 stored elements and shape (1, 88203)>
  Coords	Values
  (0, 6020)	1
  (0, 8827)	1
  (0, 19296)	1
  (0, 19340)	1
  (0, 20342)	1
  (0, 20517)	1
  (0, 34098)	1
  (0, 39784)	1
  (0, 62042)	1
  (0, 63034)	1
  (0, 64974)	1
  (0, 73015)	2
  (0, 74755)	2
  (0, 76431)	1
  (0, 80033)	1
  (0, 80424)	1


In [ ]:
dokumen = term_fit.transform(Ulasan) #hasil perhitungan tf idf dalam 1 doc
tfidf_transformer = TfidfTransformer().fit(dokumen)
print (tfidf_transformer.idf_)

tfidf=tfidf_transformer.transform(term_frequency)
print (tfidf) #hasil manual dengan sistem pyhton

[ 9.31834433  6.89761197 11.55741602 ... 13.02375309 12.73607102
 13.4292182 ]
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 16 stored elements and shape (1, 88203)>
  Coords	Values
  (0, 6020)	0.1307141172243781
  (0, 8827)	0.1485404961713671
  (0, 19296)	0.19026965724297082
  (0, 19340)	0.2574913392069116
  (0, 20342)	0.25385809524554065
  (0, 20517)	0.33072434294495817
  (0, 34098)	0.3117194713545378
  (0, 39784)	0.3679278428939415
  (0, 62042)	0.22466119746469615
  (0, 63034)	0.2661555717056662
  (0, 64974)	0.15832624738972093
  (0, 73015)	0.3734843185598679
  (0, 74755)	0.31721283418731916
  (0, 76431)	0.15489049305770156
  (0, 80033)	0.10668040582566933
  (0, 80424)	0.17354487479167777


# NLP

In [ ]:
data_label = data[["Nama_Produk", "Akun", "Ulasan_clean", "label"]]

In [ ]:
data_label["Ulasan_clean"] = data_label["Ulasan_clean"].fillna("tidak ada komentar")

In [ ]:
data_label.to_excel("data_label.xlsx")

In [ ]:
sentimen_data=pd.value_counts(data_label["label"], sort= True)
sentimen_data.plot(kind= 'bar', color= ["green", "red"])
plt.title('Bar chart')
plt.show()

Dapat dilihat bahwa isi ulasan produk lebih banyak pada label sentimen 1 atau ulasan dengan rating postitif ini berarti pelanggan yang menggunakan marketplace Tokopedia dan melakukan transaksi pembelian pada produk masker Kesehatan merasa puas bertansaksi di marketplace Tokopedia dan prosuk masker Kesehatan sehingga memberikan feedback atau ulasan komentar lebih banyak yang positif.

In [ ]:
from wordcloud import WordCloud

**Ulasan Negatif**

In [ ]:
train_s0 = data_label[data_label["label"] == 0]

In [ ]:
train_s0["Ulasan_clean"] = train_s0["Ulasan_clean"].fillna("tidak ada komentar")

In [ ]:
train_s0

In [ ]:
all_text_s0 = ' '.join(word for word in train_s0["Ulasan_clean"])
wordcloud = WordCloud(colormap='Reds', width=1000, height=1000, mode='RGBA', background_color='white').generate(all_text_s0)
plt.figure(figsize=(20,10))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")
plt.margins(x=0, y=0)
plt.show()

Dari visualisasi diatas merupakan wordcloud kata yang paling banyak muncul pada isi ulasan yang memiliki label sentimen negatif. Kata yang paling sering muncul dan mengarah ke ulasan negatif membahas seputar : barang, harga, kotak, penyok, box, dus, tipis, putus, sobek, kualitas, karet, bolong dan sebagainya. Sehingga dari kata-kata ini bisa menjadi masukan untuk penjual dan marketplace Tokopedia untuk meningkatkan kualitas barang (produk masker kesehatan), harga, kualitas pengiriman atau pengemasan, serta kualitas produk masker Kesehatan yang paling banyak disebutkan pelanggan dalam hasil Analisa ulasan sentimen yang negatif.

**Ulasan Positif**

In [ ]:
train_s1 = data_label[data_label["label"] == 1]

In [ ]:
train_s1["Ulasan_clean"] = train_s1["Ulasan_clean"].fillna("tidak ada komentar")

In [ ]:
train_s1

In [ ]:
all_text_s1 = ' '.join(word for word in train_s1["Ulasan_clean"])
wordcloud = WordCloud(colormap='Blues', width=1000, height=1000, mode='RGBA', background_color='white').generate(all_text_s1)
plt.figure(figsize=(20,10))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")
plt.title("Ulasan Positif")
plt.margins(x=0, y=0)
plt.show()

Dari visualisasi diatas merupakan wordcloud kata yang paling banyak muncul pada ulasan yang memiliki label sentimen positif. Kata yang paling sering muncul dan mengarah ke ulasan positif membahas seputar : barang, cepat, bagus, masker, aman, kualitas, sesuai, rapi, respon, aman, recommended, dan sebagainya. Sehingga dari kata-kata ini bisa menjadi masukan untuk penjual dan marketplace Tokopedia untuk menjaga kualitas atau meningkatkan kembali kualitas barang (produk masker kesehatan), kualitas yang sesuai dan aman, serta respon penjual paling banyak disebutkan pelanggan dalam hasil Analisa ulasan sentimen yang positif.

# Menyiapkan Data Train dan Test

Pada proses ini kami menggunakan library sklearn.model_selection dengan modul train_test _split untuk membagi data latih (X_train dan y_train) dan data uji (X_test dan y_test) dengan persentasi data latih 70% dan data uji 30% serta memilih label data yaitu yang merupakan variable independen dari data kami yaitu kolom label untuk dijadikan parameter klasifikasi prediksi.

In [ ]:
data_label['Ulasan_clean'] = data_label['Ulasan_clean'].fillna("tidak ada komentar")

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(data_label['Ulasan_clean'], data_label['label'],
                                                    test_size=0.1, stratify=data_label['label'], random_state=30)

# TF-IDF

Pada proses ini kami menggunakan pembobotan TF-IDF(term frequency–inverse document) untuk menghitung manual dengan menggunakan python pembobotan kata dalam dokumen data ulasan.

In [ ]:
import numpy as np

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(decode_error='replace', encoding='utf-8')

In [ ]:
X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

print(X_train.shape)
print(X_test.shape)

In [ ]:
X_train = X_train.toarray()

In [ ]:
X_test = X_test.toarray()

# Machine Learning

In [ ]:
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB()

In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold

#deklarasi metode cross validation
cv_method = RepeatedStratifiedKFold(n_splits=5,  n_repeats=3, random_state=999)
#tuning hyperparameter menggunakan gridsearch

params_NB = {'var_smoothing': np.logspace(0,-9, num=100)}
gscv_nb = GridSearchCV(estimator=nb,
                 param_grid=params_NB,
                 cv=cv_method,   # use any cross validation technique
                 verbose=1,
                 scoring='accuracy')

#Fitting ke Model
gscv_nb.fit(X_train,y_train)
#mendapatkan hyperparameters terbaik
gscv_nb.best_params_

In [ ]:
nb = GaussianNB(var_smoothing=1.0)

In [ ]:
nb.fit(X_train, y_train)

In [ ]:
y_pred_nb = nb.predict(X_test)

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [ ]:
print('--------------------- confusion matrix  ----------------------------')
print(confusion_matrix(y_test, y_pred_nb))
print('--------------------- classification report  ----------------------------')
print(classification_report(y_test, y_pred_nb))

Setelah dilakukan pembagian data latih dan data uji serta pembobotan tf-idf selanjutnya dapat dilakukan proses klasifikasi prediksi menggunakan model algoritma Naïve Bayes seperti proses yang ditunjukkan pada gambar 4. Didapatkan model algoritma Naïve Bayes dapat memberikan akurasi yang cukup baik sampari 88%.

> Dari hasil penelitian menggunakan Metode Algoritma Naïve Bayes untuk mengetahui sentimen ulasan pengguna dengan klasifikasi 2 kelas positif dan negative dengan pendekatan NLP menghasilkan nilai akurasi sebesar 88%. Selain itu, didapatkan bahwa Analisa Sentimen pada ulasan marketplace Tokopedia pada produk masker kesehatan menunjukan lebih banyak pada ulasan yang positif. Ini berarti pelayanan dan produk masker Kesehatan yang disediakan di marketplace Tokopedia sudah cukup baik.


> Dari hasil Analisis diatas dapat disimpulkan hasil scraping yang kami dapat dari produk pencarian masker kesehatan pada Tokopedia menampilkan data yang menunjukan ulasan positif lebih dominan daripada hasil ulasan negatif dan untuk ulasan negatif kata yang paling sering muncul adalah seputar kualitas produk yang tipis,mudah putus, sobek atau bolong, kualitas karetnya dan  pada pengemasan yaitu kotak/dus penyok dan untuk analisa positif kata yang paling sering muncul adalah seputar kualitas produk yang sesuai dan rapi, pengiriman yang aman dan cepat dan respon penjual.